# SIGNAL: просмотр проверяемых результатов

Этот notebook читает сохранённые эксперименты, не запускает длительное обучение. Методология: `docs/METHODOLOGY.md`. Для первичного запуска откройте `docs/PRESENTATION.pdf`.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
root = Path.cwd()
if not (root / 'pyproject.toml').exists():
    root = root.parent
assert (root / 'pyproject.toml').exists(), 'Откройте notebook из папки проекта'
results = root / 'results' / 'full'

## Парная MAE на тех же территориях

Горизонты 1/3/6/12 относятся к конкретному будущему месяцу. Для h=12 есть один целевой месяц, поэтому оценка не подтверждает устойчивое годовое качество.

In [ ]:
m = pd.read_csv(results / 'metrics_paired_prophet.csv')
test = m[(m['split'] == 'test') & (m['category'] == 'Все категории')]
test.pivot(index='model', columns='horizon', values='mae').round(1)

## Выбор собственного реального примера

Измените territory_id. Название МО не восстанавливается без официального справочника. График показывает реальные данные и сохранённые одношаговые прогнозы. Отсутствующие месяцы отмечены штриховкой; линии через пропуски не проводятся.

In [ ]:
territory_id = 1
source = pd.read_parquet(root / 'data/raw/consumption.parquet')
source['category'] = source['category'].str.strip()
source['date'] = pd.to_datetime(source.date)
dates = pd.date_range(source.date.min(), source.date.max(), freq='MS')
actual = source[(source.territory_id == territory_id) & (source.category == 'Все категории')]
actual = actual.set_index('date').value.reindex(dates)
p = pd.read_parquet(results / 'predictions_h01.parquet', filters=[('territory_id', '==', territory_id), ('model', '==', 'signal_ensemble'), ('category', '==', 'Все категории')])
forecast = p.set_index('target_date').prediction.reindex(dates)
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(dates, actual, 'o-', label='Факт', color='#263b44')
ax.plot(dates, forecast, 'D--', label='SIGNAL, h=1', color='#007d66')
for i, month in enumerate(dates[actual.isna()]):
    ax.axvspan(month-pd.Timedelta(days=14), month+pd.Timedelta(days=15),
               facecolor='#e2e5e4', edgecolor='#a8b2ad', hatch='////', alpha=.55,
               label='Нет наблюдений' if i == 0 else None)
ax.set_title(f'МО {territory_id}: средние безналичные расходы')
ax.set_ylabel('руб.')
ax.legend()
ax.grid(alpha=.15)
fig.autofmt_xdate()
plt.show()


## Детекторы: полусинтетические события

Не смешивайте эти значения с точностью предсказания реальных экономических шоков. Для реальных данных эталонных меток нет.

In [ ]:
det = pd.read_csv(results / 'detection_metrics.csv')
det.groupby(['model','kind','magnitude'])[['recall','precision','f1','false_alarms_per_100']].mean().round(3)

## Ранние предупреждения

Цель - статистически крупное относительное изменение следующего месяца, а не подтверждённый кризис. Сравнение с постоянной вероятностью показывает, есть ли предсказательная информация.

In [ ]:
pd.read_csv(results / 'early_warning_metrics.csv')

## Допустимая информация в момент решения

В файлах весов можно проверить, какие модели использовал ансамбль в каждом старте. Будущие факты не участвуют в выборе.

In [ ]:
weights = pd.read_csv(results / 'ensemble_weights.csv')
weights[(weights.category == 'Все категории') & (weights.horizon == 3) & (weights.weight > 0)]